# CT-SeqTrack four-module scratch analysis

Executed companion notebook. It reads the generated tidy CSV files and reproduces the main numerical claims without reading checkpoints.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve()
DATA = ROOT / 'compare_results' / 'data'
print('Repository:', ROOT)
print('Data:', DATA)

Repository: D:\desktop\research\CT-SeqTrack
Data: D:\desktop\research\CT-SeqTrack\compare_results\data


In [2]:
summary = pd.read_csv(DATA / 'four_scratch_validation_summary_20260725.csv')
integrity = pd.read_csv(DATA / 'four_scratch_integrity_20260725.csv')
comparisons = pd.read_csv(DATA / 'four_scratch_comparisons_20260725.csv')
summary[['run','last_validation_epoch','latest_success','latest_precision','best_success','best_precision']]

,run,last_validation_epoch,latest_success,latest_precision,best_success,best_precision
0,SeqTrack,60,50.985779,59.961712,52.283371,65.214447
1,W0,60,28.145514,27.203503,32.729759,33.176151
2,M2,60,52.578777,62.093002,52.692558,63.017509
3,M3-w0,40,49.642231,62.287754,49.642231,62.287754
4,M3-w.05,45,51.036106,55.956238,51.036106,66.775719


In [3]:
integrity[['run','status','checkpoint_epoch','latest_loss_step','last_validation_epoch','validation_points','state_tensors']]

,run,status,checkpoint_epoch,latest_loss_step,last_validation_epoch,validation_points,state_tensors
0,SeqTrack,COMPLETE,60,75719,60,12,320
1,W0,COMPLETE,60,75719,60,12,320
2,M2,COMPLETE,60,75719,60,12,334
3,M3-w0,PARTIAL,40,54796,40,8,669
4,M3-w.05,PARTIAL,45,57020,45,9,669


In [4]:
main = comparisons[(comparisons['scope'] == 'latest/final') & (comparisons['comparison'].isin(['W0 − SeqTrack','M2 − W0','M2 − SeqTrack']))]
main[['comparison','metric','delta_points','evidence']]

,comparison,metric,delta_points,evidence
0,W0 − SeqTrack,Success,-22.840265,historical cross-code comparison
1,W0 − SeqTrack,Precision,-32.758209,historical cross-code comparison
6,M2 − W0,Success,24.433264,matched current-code scratch bundle comparison
7,M2 − W0,Precision,34.889500,matched current-code scratch bundle comparison
12,M2 − SeqTrack,Success,1.592999,historical reference comparison
13,M2 − SeqTrack,Precision,2.131290,historical reference comparison


In [5]:
effects = pd.read_csv(DATA / 'four_scratch_m3_late_effects_20260725.csv')
effects[effects['metric'].isin(['M3 path loss','Irregular view B diagnostic GT loss','Canonical view A supervised loss','Center endpoint gap (m)','Yaw endpoint gap (rad)'])][['metric','M3_w0_mean','M3_w05_mean','relative_change_pct']]

,metric,M3_w0_mean,M3_w05_mean,relative_change_pct
0,M3 path loss,0.027209,0.014039,-48.402928
1,Irregular view B diagnostic GT loss,0.740525,0.525381,-29.052960
2,Canonical view A supervised loss,0.204015,0.202801,-0.595298
3,Center endpoint gap (m),0.227679,0.166927,-26.683430
4,Yaw endpoint gap (rad),0.012652,0.011728,-7.299570


In [6]:
repro = pd.read_csv(DATA / 'four_scratch_reproducibility_20260725.csv')
repro

,window,shared_steps,different_steps,different_fraction,mean_absolute_loss_difference,first_differing_step,interpretation
0,first 10 training epochs (effective M3 weight ...,12621,12612,0.999287,0.196217,9,Same nominal seed is not bitwise deterministic...


In [7]:
assert set(integrity.query("status == 'COMPLETE'")['run']) == {'SeqTrack','W0','M2'}
assert set(integrity.query("status == 'PARTIAL'")['run']) == {'M3-w0','M3-w.05'}
m2_w0 = main[(main.comparison == 'M2 − W0')].set_index('metric').delta_points
assert m2_w0['Success'] > 24 and m2_w0['Precision'] > 34
print('Integrity and headline-delta checks: PASS')

Integrity and headline-delta checks: PASS


## Interpretation guardrails

- M3 results are partial; no extrapolation to epoch60.
- Epochs are dependent observations, not statistical replicates.
- SeqTrack is a historical reference, not an exact current-code control.
- The M2 comparison identifies a bundle, not individual submodules.